<a href="https://colab.research.google.com/github/sreerajmk/railroad/blob/genAi/StockPredictionUsingGenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import datetime

# Fetch historical stock price data using yfinance
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2023, 1, 1)
stock_data = yf.download('AAPL', start=start_date, end=end_date)

# Preprocess data
stock_data = stock_data[['Close']]  # Use closing prices
stock_data = stock_data.values  # Convert to numpy array

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

# VAE model
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    dim = K.int_shape(z_mean)[1]
    epsilon = K.random_normal(shape=(batch, dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

original_dim = stock_data.shape[1]
latent_dim = 2

# Encoder
inputs = Input(shape=(original_dim,))
h = Dense(16, activation='relu')(inputs)
z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)
z = Lambda(sampling, output_shape=(latent_dim,))([z_mean, z_log_var])

# Decoder
decoder_h = Dense(16, activation='relu')
decoder_mean = Dense(original_dim, activation='sigmoid')
h_decoded = decoder_h(z)
x_decoded_mean = decoder_mean(h_decoded)

# VAE model
vae = Model(inputs, x_decoded_mean)
vae.compile(optimizer='adam', loss='mse')

# Train VAE
vae.fit(stock_data, stock_data, epochs=50, batch_size=16, shuffle=True)

# Encode data
encoder = Model(inputs, z_mean)
encoded_data = encoder.predict(stock_data)

Epoch 1/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 17480.6484
Epoch 2/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16803.5430
Epoch 3/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17457.6445
Epoch 4/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 17384.0352
Epoch 5/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16943.7637
Epoch 6/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17305.5352
Epoch 7/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17287.0488
Epoch 8/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16887.7207
Epoch 9/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16999.6973
Epoch 10/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17141.5684
Epoch 11/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16855.8926
Epoch 12/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 17166.7949
Epoch 13/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17236.1074
Epoch 14/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 17023.5664
Epoch 15/50
48/48 ━━━━━━━━━━━

In [3]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LSTM, Dropout
from tensorflow.keras.models import Sequential

# Scale data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(encoded_data)

# Prepare data for LSTM
def create_dataset(data, time_step=1):
    X, Y = [], []
    for i in range(len(data) - time_step - 1):
        a = data[i:(i + time_step), :]
        X.append(a)
        Y.append(data[i + time_step, :])
    return np.array(X), np.array(Y)

time_step = 10
X, Y = create_dataset(scaled_data, time_step)

# Reshape input to be [samples, time steps, features]
X = X.reshape(X.shape[0], X.shape[1], X.shape[2])

# LSTM model
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=Y.shape[1]))

model.compile(optimizer='adam', loss='mean_squared_error')

# Train LSTM
model.fit(X, Y, epochs=100, batch_size=32, verbose=1)

# Predict
predicted_stock_price = model.predict(X)
predicted_stock_price = scaler.inverse_transform(predicted_stock_price)


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1166
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0112
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0067
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0055
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0050
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0052
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0043
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0042
Epoch 9/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0040
Epoch 10/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0041
Epoch 11/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0040
Epoch 12/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0046
Epoch 13/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0036
Epoch 14/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0040
Epoch 15/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0038
E

In [4]:
# Print the predicted stock prices
print("Predicted Stock Prices:")
for i, price in enumerate(predicted_stock_price):
    print(f"Day {i + 1}: {price[0]:.2f}")

Predicted Stock Prices:
Day 1: -33.86
Day 2: -33.97
Day 3: -34.18
Day 4: -34.29
Day 5: -34.38
Day 6: -34.48
Day 7: -34.53
Day 8: -34.25
Day 9: -34.23
Day 10: -34.50
Day 11: -34.78
Day 12: -34.52
Day 13: -34.15
Day 14: -34.15
Day 15: -34.36
Day 16: -34.70
Day 17: -34.85
Day 18: -34.94
Day 19: -34.93
Day 20: -35.14
Day 21: -35.28
Day 22: -35.36
Day 23: -35.21
Day 24: -35.18
Day 25: -35.08
Day 26: -34.76
Day 27: -33.98
Day 28: -32.94
Day 29: -32.26
Day 30: -31.25
Day 31: -30.43
Day 32: -30.70
Day 33: -30.92
Day 34: -31.52
Day 35: -31.74
Day 36: -31.68
Day 37: -30.80
Day 38: -30.53
Day 39: -30.18
Day 40: -29.05
Day 41: -28.95
Day 42: -28.02
Day 43: -27.47
Day 44: -26.98
Day 45: -26.57
Day 46: -25.82
Day 47: -25.01
Day 48: -25.13
Day 49: -25.42
Day 50: -26.11
Day 51: -26.40
Day 52: -26.71
Day 53: -26.93
Day 54: -26.63
Day 55: -26.39
Day 56: -26.11
Day 57: -26.57
Day 58: -27.02
Day 59: -27.57
Day 60: -28.06
Day 61: -28.57
Day 62: -29.38
Day 63: -29.98
Day 64: -30.43
Day 65: -30.58
Day 66: -3